In [173]:
# ============================================================
# BLOCO 01 — CONFIGURAÇÕES
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("../")

DADOS_BRUTOS = ROOT / "dados_brutos"
DADOS_PROCESSADOS = ROOT / "dados_processados"

DADOS_PROCESSADOS.mkdir(exist_ok=True)

ARQUIVO_CRIMES = DADOS_BRUTOS / "crimes_violentos_df.xlsx"
ARQUIVO_RENDA = DADOS_BRUTOS / "dados_renda.csv"
ARQUIVO_POPULACAO = DADOS_BRUTOS / "populacao_df.csv"

ABAS = {
    "Homicídio": "homicidios",
    "Latrocínio": "latrocinios",
    "Lesão Seguida de Morte": "lesao_seguida_morte",
    "Feminicídio": "feminicidios",
}

In [174]:
# ============================================================
# BLOCO 02 — LEITURA DOS DADOS
# ============================================================

bases = {}

for aba, nome in ABAS.items():

    bases[nome] = pd.read_excel(
        ARQUIVO_CRIMES,
        sheet_name=aba,
        header=1
    )

renda = pd.read_csv(ARQUIVO_RENDA)

populacao = pd.read_csv(ARQUIVO_POPULACAO)

print(f"Bases de crimes : {len(bases)}")
print(f"Renda           : {renda.shape}")
print(f"População       : {populacao.shape}")

Bases de crimes : 4
Renda           : (36, 4)
População       : (33, 2)


In [175]:
# ============================================================
# BLOCO 03 — PADRONIZAÇÃO
# ============================================================

RENOMEAR = {
    "Região Administrativa": "regiao_administrativa"
}

for nome, df in bases.items():

    df.columns = (
        pd.Index(df.columns)
        .map(str)
        .str.strip()
    )

    df.rename(columns=RENOMEAR, inplace=True)

renda.columns = (
    renda.columns
    .str.strip()
    .str.lower()
)

populacao.columns = (
    populacao.columns
    .str.strip()
    .str.lower()
)

In [176]:
# ============================================================
# PADRONIZAÇÃO DOS NOMES DAS REGIÕES ADMINISTRATIVAS
# ============================================================

MAPA_RAS = {
    "DF Urbano": "Distrito Federal",
    "Plano Piloto": "Brasília (Plano Piloto)",
    "SCIA": "SCIA/Estrutural",
    "Sol Nascente / Pôr do Sol": "Sol Nascente/Pôr do Sol",
    "Sudoeste e Octogonal": "Sudoeste/Octogonal",
}

renda["ra"] = (
    renda["ra"]
    .replace(MAPA_RAS)
)

populacao["ra"] = (
    populacao["ra"]
    .replace(MAPA_RAS)
)

In [177]:
# ============================================================
# BLOCO 04 — TRATAMENTO
# ============================================================

ANOS = [str(ano) for ano in range(2015, 2025)]

for nome, df in bases.items():

    df[ANOS] = (
        df[ANOS]
        .replace("-", 0)
        .replace("*", np.nan)
        .apply(pd.to_numeric)
    )

    df["nivel_geografico"] = np.where(
        df["regiao_administrativa"] == "Distrito Federal",
        "Distrito Federal",
        "Região Administrativa",
    )

    df["tipo_crime"] = nome

# ============================================================
# TRATAMENTO DAS BASES AUXILIARES
# ============================================================

# Padronização dos nomes das regiões
renda["ra"] = (
    renda["ra"]
    .astype(str)
    .str.strip()
)

populacao["ra"] = (
    populacao["ra"]
    .astype(str)
    .str.strip()
)

# Criação da linha para o Distrito Federal na base de população, caso não exista
populacao = pd.concat(
    [
        populacao,
        pd.DataFrame({
            "ra": ["Distrito Federal"],
            "populacao": [2982818]   # ou o valor oficial da sua fonte
        })
    ],
    ignore_index=True
)

# Criação da dimensão territorial
dim_regioes = (
    populacao
    .merge(
        renda,
        on="ra",
        how="outer",
        validate="one_to_one"
    )
    .rename(
        columns={
            "ra": "regiao_administrativa"
        }
    )
    .sort_values(
        "regiao_administrativa",
        ignore_index=True
    )
)

In [178]:
# ============================================================
# BLOCO 05 — VALIDAÇÃO
# ============================================================

for nome, df in bases.items():

    print("=" * 60)
    print(nome.upper())

    print(df.shape)

    print(df.dtypes)

    print()

print("=" * 60)
print("RENDA")
print(renda.shape)

print("=" * 60)
print("POPULAÇÃO")
print(populacao.shape)

# ============================================================
# VALIDAÇÃO DAS BASES AUXILIARES
# ============================================================

print("=" * 60)
print("DIMENSÃO REGIÕES")

print(dim_regioes.shape)

print("\nValores ausentes")
print(dim_regioes.isna().sum())

print("\nDuplicidades")
print(
    dim_regioes["regiao_administrativa"]
    .duplicated()
    .sum()
)

HOMICIDIOS
(35, 13)
regiao_administrativa    object
2015                      int64
2016                      int64
2017                      int64
2018                      int64
2019                      int64
2020                      int64
2021                      int64
2022                      int64
2023                      int64
2024                      int64
nivel_geografico         object
tipo_crime               object
dtype: object

LATROCINIOS
(34, 13)
regiao_administrativa    object
2015                      int64
2016                      int64
2017                      int64
2018                      int64
2019                      int64
2020                      int64
2021                      int64
2022                      int64
2023                      int64
2024                      int64
nivel_geografico         object
tipo_crime               object
dtype: object

LESAO_SEGUIDA_MORTE
(34, 13)
regiao_administrativa    object
2015                      int64
2016

In [179]:
# ============================================================
# EXPORTAÇÃO
# ============================================================

for nome, df in bases.items():

    df.to_parquet(
        DADOS_PROCESSADOS / f"{nome}.parquet",
        index=False,
    )

dim_regioes.to_parquet(
    DADOS_PROCESSADOS / "dim_regioes.parquet",
    index=False,
)

print("Bases exportadas com sucesso.")

Bases exportadas com sucesso.
